# Refresh `model_benchmarks` — Artificial Analysis (Premium)

**Daily** job. Pulls the full **`/language/models`** (Pro) catalog and **overwrites** `model_benchmarks`.
The premium endpoint gives cost-per-task, a real open-weights flag, a reasoning flag, MoE params, and
richer evals directly — so this replaces the old `/data/llms/models` + LLM-Stats merge.

The biweekly `model_benchmarks_history` table is **retired** (dropped below) — no longer needed.

**Prereq:** secret `AA_API_KEY` in scope `frontier_labs` must be the **Pro** key (`aa_…`).

In [ ]:
CATALOG, SCHEMA, SCOPE, AA_SECRET = "fso_market_intelligence", "frontier_labs", "frontier_labs", "AA_API_KEY"
BASE = "https://artificialanalysis.ai/api/v2/language/models"

import requests, time
AA_KEY = dbutils.secrets.get(SCOPE, AA_SECRET)

def fetch_all():
    out, page = [], 1
    while True:
        for a in range(4):
            r = requests.get(BASE, headers={"x-api-key": AA_KEY, "Accept": "application/json"},
                             params={"page": page, "page_size": 200}, timeout=60)
            if r.status_code == 429:
                time.sleep(15 * (a + 1)); continue
            r.raise_for_status(); break
        j = r.json(); out += j.get("data", [])
        pg = j.get("pagination", {})
        if not pg.get("has_more"):
            print(f"tier={j.get('tier')} intelligence_index_version={j.get('intelligence_index_version')}")
            break
        page += 1; time.sleep(0.5)
    return out

aa = fetch_all()
print("models:", len(aa))

In [ ]:
# ── flatten EVERYTHING from /language/models (no fields dropped) ──
# Static fields are listed explicitly; evaluations + performance metrics are pulled DYNAMICALLY
# (every key present across models) so nothing is hand-picked and new AA metrics auto-appear.
def num(x):
    try: return float(x) if x is not None else None
    except (TypeError, ValueError): return None
def lng(x):
    try: return int(x) if x is not None else None
    except (TypeError, ValueError): return None

eval_keys = sorted({k for m in aa for k in (m.get("evaluations") or {})})
perf_keys = sorted({k for m in aa for k in (m.get("performance") or {})})
def evcol(k): return k.replace("artificial_analysis_", "")   # e.g. artificial_analysis_intelligence_index -> intelligence_index

rows = []
for m in aa:
    cr = m.get("model_creator") or {}
    lic = m.get("licensing") or {}
    pa = m.get("parameters") or {}
    ev = m.get("evaluations") or {}
    pr = m.get("pricing") or {}
    perf = m.get("performance") or {}
    cost = m.get("artificial_analysis_intelligence_index_cost") or {}
    cpt = cost.get("cost_per_task") or {}
    tok = m.get("artificial_analysis_intelligence_index_token_counts") or {}
    r = {
        "id": m.get("id"), "name": m.get("name"), "slug": m.get("slug"),
        "org": cr.get("name"), "country": cr.get("country"), "release_date": m.get("release_date"),
        "is_open_weights": lic.get("is_open_weights"), "reasoning_model": m.get("reasoning_model"),
        "params_total_b": num(pa.get("total")), "params_active_b": num(pa.get("active")),
        "context_window_tokens": lng(m.get("context_window_tokens")),
        # pricing ($/1M tokens) — all 6
        "price_input": num(pr.get("price_1m_input_tokens")), "price_output": num(pr.get("price_1m_output_tokens")),
        "price_blended_3to1": num(pr.get("price_1m_blended_3_to_1")),
        "price_blended_7to2to1": num(pr.get("price_1m_blended_7_to_2_to_1")),
        "price_cache_hit": num(pr.get("price_1m_cache_hit_tokens")),
        "price_cache_write": num(pr.get("price_1m_cache_write_tokens")),
        # cost per Intelligence-Index task ($) — all 4
        "cost_per_task_total": num(cpt.get("total_cost")), "cost_per_task_input": num(cpt.get("input_cost")),
        "cost_per_task_reasoning": num(cpt.get("reasoning_cost")), "cost_per_task_answer": num(cpt.get("answer_cost")),
        # full eval-suite cost ($) — all 4
        "eval_suite_cost_total": num(cost.get("total_cost")), "eval_suite_cost_input": num(cost.get("input_cost")),
        "eval_suite_cost_reasoning": num(cost.get("reasoning_cost")), "eval_suite_cost_answer": num(cost.get("answer_cost")),
        # eval-suite token counts — all 4
        "tok_input": lng(tok.get("input_tokens")), "tok_output": lng(tok.get("output_tokens")),
        "tok_reasoning": lng(tok.get("reasoning_tokens")), "tok_answer": lng(tok.get("answer_tokens")),
    }
    for k in eval_keys:                 # every evaluation (dynamic)
        r[evcol(k)] = num(ev.get(k))
    for k in perf_keys:                 # every performance metric (dynamic)
        r[k] = num(perf.get(k))
    rows.append(r)

print(f"rows: {len(rows)} | eval cols: {len(eval_keys)} | perf cols: {len(perf_keys)} | total cols: {len(rows[0])}")
print("evals:", [evcol(k) for k in eval_keys])
print("perf :", perf_keys)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DoubleType, LongType

# static fields (name, type); evals + perf appended dynamically so the schema always matches the flatten cell
static = [
    ("id", StringType), ("name", StringType), ("slug", StringType), ("org", StringType),
    ("country", StringType), ("release_date", StringType), ("is_open_weights", BooleanType),
    ("reasoning_model", BooleanType), ("params_total_b", DoubleType), ("params_active_b", DoubleType),
    ("context_window_tokens", LongType),
    ("price_input", DoubleType), ("price_output", DoubleType), ("price_blended_3to1", DoubleType),
    ("price_blended_7to2to1", DoubleType), ("price_cache_hit", DoubleType), ("price_cache_write", DoubleType),
    ("cost_per_task_total", DoubleType), ("cost_per_task_input", DoubleType),
    ("cost_per_task_reasoning", DoubleType), ("cost_per_task_answer", DoubleType),
    ("eval_suite_cost_total", DoubleType), ("eval_suite_cost_input", DoubleType),
    ("eval_suite_cost_reasoning", DoubleType), ("eval_suite_cost_answer", DoubleType),
    ("tok_input", LongType), ("tok_output", LongType), ("tok_reasoning", LongType), ("tok_answer", LongType),
]
schema = StructType([StructField(n, t()) for n, t in static])
for k in eval_keys:                      # every evaluation (dynamic — matches flatten)
    schema = schema.add(evcol(k), DoubleType())
for k in perf_keys:                      # every performance metric (dynamic)
    schema = schema.add(k, DoubleType())

df = (spark.createDataFrame(rows, schema=schema)
      .withColumn("release_date", F.to_date("release_date"))
      .withColumn("captured_at", F.current_date()))
(df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.model_benchmarks"))
print(f"model_benchmarks: {spark.table(f'{CATALOG}.{SCHEMA}.model_benchmarks').count()} rows, {len(df.columns)} cols")
print("columns:", df.columns)

# retire the old biweekly history table (no longer maintained)
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.model_benchmarks_history")
print("dropped model_benchmarks_history")

display(spark.table(f"{CATALOG}.{SCHEMA}.model_benchmarks").orderBy(F.col("intelligence_index").desc_nulls_last()).limit(10))

## Scheduling
One **daily** Job (Workflows → this notebook, **Source = Git provider / branch `main`**, serverless). The
run-as identity needs **READ** on the `frontier_labs` secret scope. Overwrites `model_benchmarks` each run.